# 最小预训练闭环 (Pre-train Minimal)

> 入口脚本：[`train/pretrain.py`](../train/pretrain.py)  
> 默认配置：[`configs/train/pretrain_tiny.yaml`](../configs/train/pretrain_tiny.yaml)

本笔记本将带你走一遍“从零训练”的代码流。我们将以 *Tiny Shakespeare* 语料为例，展示一个 LLM 预训练项目的完整生命周期。

## 1. 训练全流程数据流

```text
原始文本 (input.txt)
    │
    ▼
[ 分词器训练 ] ──► 生成 vocab.json (BPE)
    │
    ▼
[ 数据编码 ] ──► 将文本转为 uint16/uint32 的 .bin 文件 (缓存)
    │
    ▼
[ 随机采样 ] ──► 每次取 (B, T) 形状的张量
    │
    ▼
[ 模型前向 ] ──► GPT2LMHeadModel 计算 CrossEntropy Loss
    │
    ▼
[ 参数更新 ] ──► AdamW 优化器 + Cosine 学习率衰减
```

---

## 2. 关键实验环境配置

在训练脚本中，我们使用了 **OmegaConf** 来管理 YAML 配置。以下是几个核心参数的含义：

| 字段 | 作用 | 调优建议 |
| --- | --- | --- |
| `batch_size` | 显存占用核心 | 显存不够调小它，并调大 `grad_accum_steps` |
| `block_size` | 最大上下文长度 | 教学建议 128-256，正式建议 1024+ |
| `warmup_steps` | 学习率热身步数 | 避免训练初期梯度爆炸 |
| `amp` | 自动混合精度 | 在支持的 GPU 上能提速 2-3 倍 |

---

## 3. 运行命令参考

你可以直接在终端运行以下命令开启你的第一个训练任务。

### 3.1 单卡/本地调试
可以用 [`uv run`](../train/pretrain.py) 或 python 直接运行：
```powershell
python -m train.pretrain --config configs/train/pretrain_tiny.yaml
```

### 3.2 分布式 DDP (多卡并行)
如果你有多张 GPU，可以使用 `torchrun`，源码逻辑见 [`distributed.py`](../core/utils/distributed.py)：
```powershell
torchrun --nproc_per_node=2 -m train.pretrain \
    --config configs/train/pretrain_tiny.yaml distributed.backend=ddp
```

---

## 4. 训练监控与验证

### 4.1 梯度反向对比
在 [`tests/test_training_step.py`](../tests/test_training_step.py) 中，我们通过固定随机种子，验证了单步训练的 Loss 是否与预期一致。

### 4.2 采样测试
训练过程中，脚本会定期生成一段文本。对于 *Tiny Shakespeare*，你应该在运行约 200 步左右时，观察到输出开始具有明显的戏剧结构（人名 : 话语）。

---

## 5. 延伸阅读与参考资料

- **DeepSpeed 文档**: [ZeRO 优化技术](https://www.deepspeed.ai/tutorials/zero/) (如果你想训练 1B+ 的模型)
- **FineWeb 数据集**: [HuggingFace FineWeb](https://huggingface.co/spaces/HuggingFaceFW/blogpost-fineweb-v1) (目前开源界最强预训练语料库)

---
> [03 · Attention](03_attention_mha.ipynb) | [README.ipynb](README.ipynb)
